# Using FFTLog

In this tutorial, we will reproduce the example in appendix B of [[astro-ph/9905191]](https://arxiv.org/abs/astro-ph/9905191). See also the [webpage](https://jila.colorado.edu/~ajsh/FFTLog/).

## Running the notebook

If you use the `uv` package manager, you need to register its virtual environment as a Jupyter kernel, which can be done by running the following command in the project root:

```shell
uv run --group dev --group tutorial ipython kernel install --user --env VIRTUAL_ENV $(pwd)/.venv --name=project
```

See the [`uv` docs](https://docs.astral.sh/uv/guides/integration/jupyter/#using-jupyter-within-a-project) for more details.

In [ ]:
import camb
import numpy as np
import scipy.special
from fftloggin import fht, ifht, fhtoffset
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = 'retina'

We use CAMB to generate the matter power spectrum:

In [ ]:
h = 0.675
minkh = 1e-4
maxkh = 1
npoints = 512
dln = np.log(maxkh / minkh) / (npoints - 1)
k_per_logint = int(1 / dln)

redshift_slices = [0, 1, 2]
params = camb.set_params(
    H0=100 * h,
    ombh2=0.022,
    omch2=0.122,
    mnu=0.06,
    omk=0,
    tau=0.06,
    As=2e-9,
    ns=0.965,
    halofit_version="mead",
    lmax=3000,
)
params.set_matter_power(redshifts=redshift_slices, kmax=maxkh / h, k_per_logint=k_per_logint)
# calculate results for these parameters
results = camb.get_results(params)

In [ ]:
kh, z, pk = results.get_matter_power_spectrum(minkh=minkh, maxkh=maxkh, npoints=npoints)

fig, ax = plt.subplots()
for i, redshift in enumerate(redshift_slices):
    ax.loglog(kh, pk[i, :], label=rf"$z={redshift}$")
ax.set_xlabel(r"$k \, \rm{[h/Mpc]}$")
ax.set_ylabel(r"$P(k) \, \rm{[Mpc/h]^3}$")
ax.set_title("Matter Power Spectrum")
ax.legend()
ax.grid()

## The Hankel transform

Converting the power spectrum into the real-space correlation function $\xi(r)$ amounts to taking the Fourier transform,

$$ \xi(r) = \int d^3k P(k)e^{i \bm{k} \cdot \bm{r}}$$

The radial-angular split collapses the plane wave $e^{i \bm{k} \cdot \bm{r}}$ into a volume factor and a spherical Bessel function,

$$
\begin{align*}
\xi(r) &= \frac{1}{2\pi^2}\int dk k^2 P(k) j_0(kr) \\
&= \frac{1}{2\pi^2}\int dk k^2 P(k) \frac{\sin (kr)}{kr}.
\end{align*}
$$

The inverse Hankel transform that the code computes has a slightly different definition. For a transform pair $a(r), \tilde{a}(k)$,

$$
\begin{align*}
\tilde{a}(k) &= \int J_\mu(kr)a(r)kdr,\\
a(r) &= \int J_\mu(kr)a(k)rdk,
\end{align*}
$$

where $J_\mu(\cdot)$ is the Bessel function of order $\mu$, which is related to the spherical bessel function $j_\mu$ by

$$j_\mu(x) = \sqrt{\frac{\pi}{2x}}J_{\mu + 1/2}(x).$$

Recasting the expression of the correlation function as an explicit Hankel transform,

$$
\begin{align*}
\xi(r) &= \frac{1}{2\pi^2}\int k^2 P(k) \sqrt{\frac{\pi}{2kr}}J_{1/2}(kr)dk \\
&= \frac{1}{(2\pi r)^{3/2}}\int k^{3/2} P(k) J_{1/2}(kr)rdk\\
&= \frac{1}{(2\pi r)^{3/2}} \rm{IFHT} [k^{3/2} P(k), \mu=1/2]
\end{align*}
$$

In the cell below, we compute the correlation function $\xi(r)$ via the inverse Hankel transform using the FFTLog algorithm. We also compute the integration directly for comparison.

In [ ]:
ic = (npoints - 1) // 2
i = np.arange(npoints)

def get_r(kh: np.ndarray, dln: float, offset: float) -> np.ndarray:
    ic = (npoints - 1) // 2
    return np.exp((i - ic) * dln + offset) / kh[ic]

def get_xi_fftlog(kh: np.ndarray, r: np.ndarray, pk: np.ndarray, dln: float, offset: float):
    xi_r = ifht(kh**(3/2) * pk, dln=dln, mu=0.5, offset=offset)
  # offset parameter is set to 0, which implies (rk)_0 = 1, where index 0 is the central element.
    xi_r /= (2 * np.pi * r) ** (3/2)
    return xi_r

def get_xi_direct_integration(kh: np.ndarray, r: np.ndarray, pk: np.ndarray):
  log_kh = np.log(kh)
  integrand = pk * kh**3 * scipy.special.spherical_jn(0, np.outer(r, kh))
  return np.trapezoid(integrand, x=log_kh, axis=-1) / (2 * np.pi**2)

In [ ]:
fig, ax = plt.subplots()

r = get_r(kh, dln=dln, offset=0.)
xi_r = get_xi_fftlog(kh, r, pk[0, :], dln=dln, offset=0.)
xi_direct_integration = get_xi_direct_integration(kh, r, pk[0, :])
ax.loglog(r, xi_r, label="FFTLog")
ax.loglog(r, xi_direct_integration, label="Direct Integration")
ax.set_xlabel(r"$r \, \rm{[Mpc/h]}$")
ax.set_ylabel(r"$\xi(r)$")
ax.legend()

Performance benchmarking:

In [ ]:
%timeit -q -v fftlog_time get_xi_fftlog(kh, r, pk[0, :], dln=dln, offset=0.)
%timeit -q -v direct_integration_time get_xi_direct_integration(kh, r, pk[0, :])
print("FFTLog:", fftlog_time)
print("Direct integration:", direct_integration_time)
speedup = direct_integration_time.average / fftlog_time.average
print(f"\tAverage speedup: {speedup:.2f}x")

## Minimizing ringing

In [ ]:
bias = -0.3
offset = fhtoffset(dln, mu=0.5, initial=0, bias=bias)

print("Offset =", offset)
xi_r = ifht(kh** (3/2) * pk[0, :], dln=dln, mu=0.5, offset=offset, bias=bias)

# offset parameter is set to 0, which implies (rk)_0 = 1, where index 0 is the central element.
ic = (npoints - 1) // 2
i = np.arange(npoints)
r = np.exp((i - ic) * dln + offset) / kh[ic]

xi_r = xi_r / (2 * np.pi * r) ** (3/2)

In [ ]:
fig, ax = plt.subplots()

ax.loglog(r, xi_r, label="FFTLog")
ax.loglog(r, xi_direct_integration, label="Direct Integration")
ax.set_xlabel(r"$r \, \rm{[Mpc/h]}$")
ax.set_ylabel(r"$\xi(r)$")
ax.set_title(f"Offset={offset:.1e}, bias={bias:.2f}")
ax.legend()